# PAYMENTS CLEANING

In [266]:
import pandas as pd
payments = pd.read_csv("../Data/sanlam_raw_payments.csv")

payments.head()

,source_row_id,transaction_ref,policy_id,client_id,transaction_date,amount,transaction_type,currency,payment_method,payment_status,source_system
0,SPAY-0000000001,TX-S-0000000001,POL000003392,CL00007794,2022-11-15,"1,826.09",Commission,R,NaN,Reversed,LegacyLedger
1,SPAY-0000000002,TX-S-0000000002,POL000000796,CL00008081,2022-10-23,"R -7,370.85",Refund,ZAR,EFT,success,PaymentsHub
2,SPAY-0000000003,TX-S-0000000003,POL000003511,CL00007728,10/03/2025,"R 4,318.19",EFT,ZAR,Card,Paid,LegacyLedger
3,SPAY-0000000004,TX-S-0000000004,POL000012172,CL00001221,2025/03/21,"R 4,114.14",Debit Order,R,EFT,Successful,PaymentsHub
4,SPAY-0000000005,TX-S-0000000005,POL000014430,CL00008773,2025/04/23,"ZAR 21,151.57",EFT,ZAR,Debit Order,Reversed,BankFeed


In [267]:
# Check to see if the transaction date has any NA values 
payments['transaction_date'].isna().sum()


np.int64(162)

In [268]:
# Step 1: Create a parser for the known formats
def parse_transaction_date(date_str):
    if pd.isna(date_str):
        return pd.NaT

    date_str = str(date_str).strip()

    try:
        # DD/MM/YYYY
        if '/' in date_str and len(date_str.split('/')[0]) == 2:
            return pd.to_datetime(date_str, format='%d/%m/%Y')

        # MM-DD-YYYY
        elif '-' in date_str and len(date_str.split('-')[0]) == 2:
            return pd.to_datetime(date_str, format='%m-%d-%Y')

        # YYYY-MM-DD
        elif len(date_str.split('-')[0]) == 4:
            return pd.to_datetime(date_str, format='%Y-%m-%d')

        # YYYY/MM/DD
        elif len(date_str.split('/')[0]) == 4:
            return pd.to_datetime(date_str, format='%Y/%m/%d')

        return pd.NaT

    except:
        return pd.NaT

In [269]:
# Step 2: Parse the dates
payments['parse_transaction_date'] = (
    payments['transaction_date']
    .apply(parse_transaction_date)
)

In [270]:
payments.head()

,source_row_id,transaction_ref,policy_id,client_id,transaction_date,amount,transaction_type,currency,payment_method,payment_status,source_system,parse_transaction_date
0,SPAY-0000000001,TX-S-0000000001,POL000003392,CL00007794,2022-11-15,"1,826.09",Commission,R,NaN,Reversed,LegacyLedger,2022-11-15
1,SPAY-0000000002,TX-S-0000000002,POL000000796,CL00008081,2022-10-23,"R -7,370.85",Refund,ZAR,EFT,success,PaymentsHub,2022-10-23
2,SPAY-0000000003,TX-S-0000000003,POL000003511,CL00007728,10/03/2025,"R 4,318.19",EFT,ZAR,Card,Paid,LegacyLedger,2025-03-10
3,SPAY-0000000004,TX-S-0000000004,POL000012172,CL00001221,2025/03/21,"R 4,114.14",Debit Order,R,EFT,Successful,PaymentsHub,2025-03-21
4,SPAY-0000000005,TX-S-0000000005,POL000014430,CL00008773,2025/04/23,"ZAR 21,151.57",EFT,ZAR,Debit Order,Reversed,BankFeed,2025-04-23


In [271]:
# Step 3: Find malformed dates
bad_dates = payments[
    payments['transaction_date'].notna() &
    payments['parse_transaction_date'].isna()
]

In [272]:
# Step 4: Count malformed dates
bad_dates.shape[0]

454

In [273]:
#Step 5
payments['transaction_date_std'] = (
    payments['parse_transaction_date']
    .dt.strftime('%d-%m-%Y')
)

In [274]:
payments.columns.tolist()

['source_row_id',
 'transaction_ref',
 'policy_id',
 'client_id',
 'transaction_date',
 'amount',
 'transaction_type',
 'currency',
 'payment_method',
 'payment_status',
 'source_system',
 'parse_transaction_date',
 'transaction_date_std']

In [275]:
payments.head(30)

,source_row_id,transaction_ref,policy_id,client_id,transaction_date,amount,transaction_type,currency,payment_method,payment_status,source_system,parse_transaction_date,transaction_date_std
0,SPAY-0000000001,TX-S-0000000001,POL000003392,CL00007794,2022-11-15,"1,826.09",Commission,R,NaN,Reversed,LegacyLedger,2022-11-15,15-11-2022
1,SPAY-0000000002,TX-S-0000000002,POL000000796,CL00008081,2022-10-23,"R -7,370.85",Refund,ZAR,EFT,success,PaymentsHub,2022-10-23,23-10-2022
2,SPAY-0000000003,TX-S-0000000003,POL000003511,CL00007728,10/03/2025,"R 4,318.19",EFT,ZAR,Card,Paid,LegacyLedger,2025-03-10,10-03-2025
3,SPAY-0000000004,TX-S-0000000004,POL000012172,CL00001221,2025/03/21,"R 4,114.14",Debit Order,R,EFT,Successful,PaymentsHub,2025-03-21,21-03-2025
4,SPAY-0000000005,TX-S-0000000005,POL000014430,CL00008773,2025/04/23,"ZAR 21,151.57",EFT,ZAR,Debit Order,Reversed,BankFeed,2025-04-23,23-04-2025
5,SPAY-0000000006,TX-S-0000000006,POL000008647,CL00000156,03-09-2025,NaN,Premium,USD,NaN,Failed,BankFeed,2025-03-09,09-03-2025
6,SPAY-0000000007,TX-S-0000000007,POL000000436,CL00003805,2024-06-23,"(-24,653.54)",Reversal,R,Cash,Successful,PaymentsHub,2024-06-23,23-06-2024
7,SPAY-0000000008,TX-S-0000000008,POL000024244,CL00000252,31/05/2022,5285.68,Premium,ZAR,Cash,Paid,PaymentsHub,2022-05-31,31-05-2022
8,SPAY-0000000009,TX-S-0000000009,POL000012945,CL00007328,31/03/2021,"20,646.63",Premium,ZAR,Debit Order,Reversed,LegacyLedger,2021-03-31,31-03-2021
9,SPAY-0000000010,TX-S-0000000010,POL000010985,CL00004931,2025-10-10,"ZAR -21,763.22",Commission Reversal,R,Card,Successful,BankFeed,2025-10-10,10-10-2025


In [276]:
#Rename the transaction_date_std to transaction_date_new
payments.rename(columns={'transaction_date_std': 'transaction_date_new'}, inplace=True)

In [277]:
# Remove the parse_transaction_date coloumn 
payments.drop(columns=['parse_transaction_date'], inplace=True)

In [278]:
# Remove the transaction_date coloumn (original)
payments.drop(columns=['transaction_date'], inplace=True)

In [279]:
cols = payments.columns.tolist()

# Remove the column from its current position
cols.remove('transaction_date_new')

# Insert it into the 4th position
cols.insert(3, 'transaction_date_new')

# Reorder the dataframe
payments = payments[cols]

In [280]:
payments.dtypes

source_row_id           str
transaction_ref         str
policy_id               str
transaction_date_new    str
client_id               str
amount                  str
transaction_type        str
currency                str
payment_method          str
payment_status          str
source_system           str
dtype: object

In [281]:
# Convert string to datetime 
payments['transaction_date_new'] = pd.to_datetime(
    payments['transaction_date_new'],
    format='%d-%m-%Y',
    errors='coerce'
)

In [282]:
## Check the malformed dates (QUARTINIE ALL 616 ROWS)
payments['transaction_date_new'].isna().sum()

np.int64(616)

In [283]:
# Check the currency distinct values 
payments['currency'].unique().tolist()

['R', 'ZAR', 'USD']

In [284]:
payments['currency'] = payments['currency'].replace('R', 'ZAR')

In [285]:
# Check the distinct values for payment method. 
payments['payment_method'].unique().tolist()

[nan, 'EFT', 'Card', 'Debit Order', 'Cash']

In [286]:
payments['payment_method'].isna().sum()

np.int64(12020)

In [287]:
# Set 1202payment_method0 payment method to "Unknown"
payments['payment_method'] = payments['payment_method'].fillna('Unknown')

In [288]:
payments['payment_method'].head(10)

0        Unknown
1            EFT
2           Card
3            EFT
4    Debit Order
5        Unknown
6           Cash
7           Cash
8    Debit Order
9           Card
Name: payment_method, dtype: str

In [289]:
payments['payment_method'].unique().tolist()

['Unknown', 'EFT', 'Card', 'Debit Order', 'Cash']

In [290]:
#Remove trailling spaces and whitespaces. 
payments['payment_method'] = payments['payment_method'].str.strip()

In [291]:
payments.head(10)

,source_row_id,transaction_ref,policy_id,transaction_date_new,client_id,amount,transaction_type,currency,payment_method,payment_status,source_system
0,SPAY-0000000001,TX-S-0000000001,POL000003392,2022-11-15,CL00007794,"1,826.09",Commission,ZAR,Unknown,Reversed,LegacyLedger
1,SPAY-0000000002,TX-S-0000000002,POL000000796,2022-10-23,CL00008081,"R -7,370.85",Refund,ZAR,EFT,success,PaymentsHub
2,SPAY-0000000003,TX-S-0000000003,POL000003511,2025-03-10,CL00007728,"R 4,318.19",EFT,ZAR,Card,Paid,LegacyLedger
3,SPAY-0000000004,TX-S-0000000004,POL000012172,2025-03-21,CL00001221,"R 4,114.14",Debit Order,ZAR,EFT,Successful,PaymentsHub
4,SPAY-0000000005,TX-S-0000000005,POL000014430,2025-04-23,CL00008773,"ZAR 21,151.57",EFT,ZAR,Debit Order,Reversed,BankFeed
5,SPAY-0000000006,TX-S-0000000006,POL000008647,2025-03-09,CL00000156,NaN,Premium,USD,Unknown,Failed,BankFeed
6,SPAY-0000000007,TX-S-0000000007,POL000000436,2024-06-23,CL00003805,"(-24,653.54)",Reversal,ZAR,Cash,Successful,PaymentsHub
7,SPAY-0000000008,TX-S-0000000008,POL000024244,2022-05-31,CL00000252,5285.68,Premium,ZAR,Cash,Paid,PaymentsHub
8,SPAY-0000000009,TX-S-0000000009,POL000012945,2021-03-31,CL00007328,"20,646.63",Premium,ZAR,Debit Order,Reversed,LegacyLedger
9,SPAY-0000000010,TX-S-0000000010,POL000010985,2025-10-10,CL00004931,"ZAR -21,763.22",Commission Reversal,ZAR,Card,Successful,BankFeed


In [292]:
# Check if the coloumns contain R or ZAR 
payments[
    payments['amount'].astype(str).str.contains('R|ZAR', na=False)
]

,source_row_id,transaction_ref,policy_id,transaction_date_new,client_id,amount,transaction_type,currency,payment_method,payment_status,source_system
1,SPAY-0000000002,TX-S-0000000002,POL000000796,2022-10-23,CL00008081,"R -7,370.85",Refund,ZAR,EFT,success,PaymentsHub
2,SPAY-0000000003,TX-S-0000000003,POL000003511,2025-03-10,CL00007728,"R 4,318.19",EFT,ZAR,Card,Paid,LegacyLedger
3,SPAY-0000000004,TX-S-0000000004,POL000012172,2025-03-21,CL00001221,"R 4,114.14",Debit Order,ZAR,EFT,Successful,PaymentsHub
4,SPAY-0000000005,TX-S-0000000005,POL000014430,2025-04-23,CL00008773,"ZAR 21,151.57",EFT,ZAR,Debit Order,Reversed,BankFeed
9,SPAY-0000000010,TX-S-0000000010,POL000010985,2025-10-10,CL00004931,"ZAR -21,763.22",Commission Reversal,ZAR,Card,Successful,BankFeed
...,...,...,...,...,...,...,...,...,...,...,...
59985,SPAY-0000059986,TX-S-0000059986,POL000007834,2023-01-23,CL00009289,"ZAR 15,542.75",Premium,ZAR,Card,Reversed,PaymentsHub
59986,SPAY-0000059987,TX-S-0000059987,POL000008737,2020-08-09,CL00003597,"ZAR 18,075.50",Premium,ZAR,Cash,Paid,BankFeed
59990,SPAY-0000059991,TX-S-0000059991,POL000009315,2025-08-26,CL00004535,"ZAR 2,172.71",Commission,ZAR,Unknown,Reversed,LegacyLedger
59996,SPAY-0000059997,TX-S-0000059997,POL000023639,2024-12-05,CL00007282,"ZAR 9,196.56",Commission,ZAR,Cash,Failed,PaymentsHub


In [293]:
#Replace the R and ZAR from amounts 
payments['amount'] = payments['amount'].str.replace('ZAR', '', regex=False)
payments['amount'] = payments['amount'].str.replace('R', '', regex=False)
payments['amount'] = payments['amount'].str.strip()

In [294]:
#Check the distinct values in transcation type
payments['transaction_type'].unique().tolist()

['Commission',
 'Refund',
 'EFT',
 'Debit Order',
 'Premium',
 'Reversal',
 'Commission Reversal']

In [295]:
#Display the value counts for each transaction type
payments['payment_status'].value_counts(dropna=False)

payment_status
Successful    10117
success       10039
Reversed      10020
Failed         9991
Paid           9963
Pending        9870
Name: count, dtype: int64

In [296]:
# Merge success into successful. 
payments['payment_status'] = payments['payment_status'].replace({
    'success': 'Successful'
})

In [297]:
# Merge paid into Successful
payments['payment_status'] = payments['payment_status'].replace({
    'Paid': 'Successful'
})

In [298]:
payments['transaction_type'] = payments['transaction_type'].str.strip()

In [299]:
payments.columns

Index(['source_row_id', 'transaction_ref', 'policy_id', 'transaction_date_new',
       'client_id', 'amount', 'transaction_type', 'currency', 'payment_method',
       'payment_status', 'source_system'],
      dtype='str')

In [300]:
#Create a status crosswalk. 
status_crosswalk = {
    'success': 'Successful',
    'Successful': 'Successful',
    'Paid': 'Paid',
    'Reversed': 'Reversed',
    'Failed': 'Failed',
    'Pending': 'Pending'
}

In [301]:
payments['payment_status'] = (
    payments['payment_status']
    .map(status_crosswalk)
)

In [302]:
crosswalk = pd.DataFrame({
    'original_value': ['success', 'Successful', 'Paid', 'Reversed', 'Failed', 'Pending'],
    'standardized_value': ['Successful', 'Successful', 'Paid', 'Reversed', 'Failed', 'Pending']
})

print(crosswalk)

  original_value standardized_value
0        success         Successful
1     Successful         Successful
2           Paid               Paid
3       Reversed           Reversed
4         Failed             Failed
5        Pending            Pending


In [303]:
#Create a backup if anything goes wrong 
payments_backup = payments.copy()

In [304]:
# Created a third backup 
payments_v3 = payments_backup.copy() 

In [305]:
payments.head(10)

,source_row_id,transaction_ref,policy_id,transaction_date_new,client_id,amount,transaction_type,currency,payment_method,payment_status,source_system
0,SPAY-0000000001,TX-S-0000000001,POL000003392,2022-11-15,CL00007794,"1,826.09",Commission,ZAR,Unknown,Reversed,LegacyLedger
1,SPAY-0000000002,TX-S-0000000002,POL000000796,2022-10-23,CL00008081,"-7,370.85",Refund,ZAR,EFT,Successful,PaymentsHub
2,SPAY-0000000003,TX-S-0000000003,POL000003511,2025-03-10,CL00007728,"4,318.19",EFT,ZAR,Card,Successful,LegacyLedger
3,SPAY-0000000004,TX-S-0000000004,POL000012172,2025-03-21,CL00001221,"4,114.14",Debit Order,ZAR,EFT,Successful,PaymentsHub
4,SPAY-0000000005,TX-S-0000000005,POL000014430,2025-04-23,CL00008773,"21,151.57",EFT,ZAR,Debit Order,Reversed,BankFeed
5,SPAY-0000000006,TX-S-0000000006,POL000008647,2025-03-09,CL00000156,NaN,Premium,USD,Unknown,Failed,BankFeed
6,SPAY-0000000007,TX-S-0000000007,POL000000436,2024-06-23,CL00003805,"(-24,653.54)",Reversal,ZAR,Cash,Successful,PaymentsHub
7,SPAY-0000000008,TX-S-0000000008,POL000024244,2022-05-31,CL00000252,5285.68,Premium,ZAR,Cash,Successful,PaymentsHub
8,SPAY-0000000009,TX-S-0000000009,POL000012945,2021-03-31,CL00007328,"20,646.63",Premium,ZAR,Debit Order,Reversed,LegacyLedger
9,SPAY-0000000010,TX-S-0000000010,POL000010985,2025-10-10,CL00004931,"-21,763.22",Commission Reversal,ZAR,Card,Successful,BankFeed


In [306]:
#Counts if the transcation ref appears more than once 
payments['transaction_ref'].value_counts()[payments['transaction_ref'].value_counts() > 1]

transaction_ref
TX-S-0000003579    4
TX-S-0000000069    3
TX-S-0000000072    3
TX-S-0000000347    3
TX-S-0000000441    3
                  ..
TX-S-0000051281    2
TX-S-0000051967    2
TX-S-0000052514    2
TX-S-0000053824    2
TX-S-0000056554    2
Name: count, Length: 696, dtype: int64

In [307]:
payments.loc[
    payments['policy_id'].astype(str).str.contains('MISSING', na=False)
]

,source_row_id,transaction_ref,policy_id,transaction_date_new,client_id,amount,transaction_type,currency,payment_method,payment_status,source_system
16,SPAY-0000000017,TX-S-0000000017,MISSING-859967,2022-09-06,CL00001008,"14,281.58",EFT,ZAR,EFT,Pending,LegacyLedger
19,SPAY-0000000020,TX-S-0000000020,MISSING-584620,2022-03-22,CL00006573,"11,316.51",EFT,ZAR,Debit Order,Successful,BankFeed
37,SPAY-0000000038,TX-S-0000000038,MISSING-745650,2025-09-18,MISSING-891886,1597.36,Premium,USD,EFT,Successful,PaymentsHub
179,SPAY-0000000180,TX-S-0000000180,MISSING-527227,2025-04-06,CL00007106,"9,184.44",Commission,ZAR,Debit Order,Reversed,LegacyLedger
236,SPAY-0000000237,TX-S-0000000237,MISSING-845014,2020-10-17,CL00001024,"10,844.02",EFT,ZAR,Unknown,Pending,PaymentsHub
...,...,...,...,...,...,...,...,...,...,...,...
59751,SPAY-0000059752,TX-S-0000059752,MISSING-269132,2020-01-27,CL00009506,"(-17,539.46)",Reversal,ZAR,Card,Successful,PaymentsHub
59770,SPAY-0000059771,TX-S-0000059771,MISSING-220092,2022-08-05,CL00003796,"11,561.27",Commission,USD,Debit Order,Reversed,LegacyLedger
59849,SPAY-0000059850,TX-S-0000059850,MISSING-402395,2022-08-27,CL00011178,"16,576.43",Commission,ZAR,Debit Order,Failed,LegacyLedger
59925,SPAY-0000059926,TX-S-0000059926,MISSING-126451,2024-01-01,CL00001410,"1,565.66",Premium,ZAR,Unknown,Successful,BankFeed


In [308]:
semi_clean = payments.to_csv("../semi-clean/semi_clean_payments.csv", index = False)

In [309]:
#payments['policy_id_flag'] = payments['policy_id'].isin(['MISSING', 'MISSING-'])

In [310]:
payments['A04_flag'] = 'False'
 
payments.loc[
    payments['policy_id'].str.startswith('MISSING-', na=False),
    'A04_flag'
] = 'True'
 

In [311]:
#Remove trailling spaces and whitespaces from amount
payments['amount'] = payments['amount'].str.strip()

In [312]:
# Convert object to numeric for amount
payments['amount'] = pd.to_numeric(payments['amount'], errors='coerce') 

In [313]:
payments.head()

,source_row_id,transaction_ref,policy_id,transaction_date_new,client_id,amount,transaction_type,currency,payment_method,payment_status,source_system,A04_flag
0,SPAY-0000000001,TX-S-0000000001,POL000003392,2022-11-15,CL00007794,NaN,Commission,ZAR,Unknown,Reversed,LegacyLedger,False
1,SPAY-0000000002,TX-S-0000000002,POL000000796,2022-10-23,CL00008081,NaN,Refund,ZAR,EFT,Successful,PaymentsHub,False
2,SPAY-0000000003,TX-S-0000000003,POL000003511,2025-03-10,CL00007728,NaN,EFT,ZAR,Card,Successful,LegacyLedger,False
3,SPAY-0000000004,TX-S-0000000004,POL000012172,2025-03-21,CL00001221,NaN,Debit Order,ZAR,EFT,Successful,PaymentsHub,False
4,SPAY-0000000005,TX-S-0000000005,POL000014430,2025-04-23,CL00008773,NaN,EFT,ZAR,Debit Order,Reversed,BankFeed,False


In [314]:
payments[payments['amount'].isna()]

,source_row_id,transaction_ref,policy_id,transaction_date_new,client_id,amount,transaction_type,currency,payment_method,payment_status,source_system,A04_flag
0,SPAY-0000000001,TX-S-0000000001,POL000003392,2022-11-15,CL00007794,NaN,Commission,ZAR,Unknown,Reversed,LegacyLedger,False
1,SPAY-0000000002,TX-S-0000000002,POL000000796,2022-10-23,CL00008081,NaN,Refund,ZAR,EFT,Successful,PaymentsHub,False
2,SPAY-0000000003,TX-S-0000000003,POL000003511,2025-03-10,CL00007728,NaN,EFT,ZAR,Card,Successful,LegacyLedger,False
3,SPAY-0000000004,TX-S-0000000004,POL000012172,2025-03-21,CL00001221,NaN,Debit Order,ZAR,EFT,Successful,PaymentsHub,False
4,SPAY-0000000005,TX-S-0000000005,POL000014430,2025-04-23,CL00008773,NaN,EFT,ZAR,Debit Order,Reversed,BankFeed,False
...,...,...,...,...,...,...,...,...,...,...,...,...
59993,SPAY-0000059994,TX-S-0000059994,POL000025924,2025-07-08,CL00011070,NaN,Commission,ZAR,Unknown,Successful,LegacyLedger,False
59995,SPAY-0000059996,TX-S-0000059996,POL000014233,2024-10-07,CL00008192,NaN,EFT,USD,Debit Order,Successful,PaymentsHub,False
59996,SPAY-0000059997,TX-S-0000059997,POL000023639,2024-12-05,CL00007282,NaN,Commission,ZAR,Cash,Failed,PaymentsHub,False
59997,SPAY-0000059998,TX-S-0000059998,POL000022106,2020-11-17,CL00003267,NaN,Debit Order,ZAR,Cash,Pending,PaymentsHub,False


# PAYMENTS_BACKUP 

In [315]:
payments_backup.head(10)

,source_row_id,transaction_ref,policy_id,transaction_date_new,client_id,amount,transaction_type,currency,payment_method,payment_status,source_system
0,SPAY-0000000001,TX-S-0000000001,POL000003392,2022-11-15,CL00007794,"1,826.09",Commission,ZAR,Unknown,Reversed,LegacyLedger
1,SPAY-0000000002,TX-S-0000000002,POL000000796,2022-10-23,CL00008081,"-7,370.85",Refund,ZAR,EFT,Successful,PaymentsHub
2,SPAY-0000000003,TX-S-0000000003,POL000003511,2025-03-10,CL00007728,"4,318.19",EFT,ZAR,Card,Successful,LegacyLedger
3,SPAY-0000000004,TX-S-0000000004,POL000012172,2025-03-21,CL00001221,"4,114.14",Debit Order,ZAR,EFT,Successful,PaymentsHub
4,SPAY-0000000005,TX-S-0000000005,POL000014430,2025-04-23,CL00008773,"21,151.57",EFT,ZAR,Debit Order,Reversed,BankFeed
5,SPAY-0000000006,TX-S-0000000006,POL000008647,2025-03-09,CL00000156,NaN,Premium,USD,Unknown,Failed,BankFeed
6,SPAY-0000000007,TX-S-0000000007,POL000000436,2024-06-23,CL00003805,"(-24,653.54)",Reversal,ZAR,Cash,Successful,PaymentsHub
7,SPAY-0000000008,TX-S-0000000008,POL000024244,2022-05-31,CL00000252,5285.68,Premium,ZAR,Cash,Successful,PaymentsHub
8,SPAY-0000000009,TX-S-0000000009,POL000012945,2021-03-31,CL00007328,"20,646.63",Premium,ZAR,Debit Order,Reversed,LegacyLedger
9,SPAY-0000000010,TX-S-0000000010,POL000010985,2025-10-10,CL00004931,"-21,763.22",Commission Reversal,ZAR,Card,Successful,BankFeed


In [316]:
payments_backup['A04_flag'] = 'False'
 
payments_backup.loc[
    payments_backup['policy_id'].str.startswith('MISSING-', na=False),
    'A04_flag'
] = 'True'

In [317]:
payments_backup.dtypes

source_row_id                      str
transaction_ref                    str
policy_id                          str
transaction_date_new    datetime64[us]
client_id                          str
amount                             str
transaction_type                   str
currency                           str
payment_method                     str
payment_status                     str
source_system                      str
A04_flag                           str
dtype: object

In [318]:
payments_backup['transaction_date_new'].isnull()

0        False
1        False
2        False
3        False
4        False
         ...  
59995    False
59996    False
59997    False
59998    False
59999    False
Name: transaction_date_new, Length: 60000, dtype: bool

In [319]:
# Create quarantine dataset
quarantine_date = payments_backup[payments_backup['transaction_date_new'].isna()]
 
# Remaining valid records
payments_cleaned = payments_backup[payments_backup['transaction_date_new'].notna()]

In [320]:
# Checking if the transaction date  is correctly quarantined 
print("Original records:", len(payments_cleaned))
print("Quarantined records:", len(quarantine_date))
print('Total Records:', len(quarantine_date)+ len(payments_cleaned))
 

Original records: 59384
Quarantined records: 616
Total Records: 60000


In [321]:
#Creating a new table for the quarantined fsp number
quarantine_date.to_csv("../quarantine/quarantine_date.csv", index=False)

In [322]:
#Check if all the quaratined date values have moved over to the quarantine file. 
quarantine_date

,source_row_id,transaction_ref,policy_id,transaction_date_new,client_id,amount,transaction_type,currency,payment_method,payment_status,source_system,A04_flag
60,SPAY-0000000061,TX-S-0000000061,POL000002607,NaT,CL00007481,"2,671.52",Debit Order,ZAR,Unknown,Successful,PaymentsHub,False
104,SPAY-0000000105,TX-S-0000000105,POL000010865,NaT,CL00002170,6054.51,Debit Order,ZAR,Card,Pending,LegacyLedger,False
112,SPAY-0000000113,TX-S-0000000113,POL000013472,NaT,CL00002934,497.45,Debit Order,ZAR,Debit Order,Successful,PaymentsHub,False
146,SPAY-0000000147,TX-S-0000000147,POL000019919,NaT,CL00008635,"18,144.84",Premium,ZAR,EFT,Successful,LegacyLedger,False
181,SPAY-0000000182,TX-S-0000000182,POL000003624,NaT,CL00000134,255.14,Premium,ZAR,Debit Order,Successful,PaymentsHub,False
...,...,...,...,...,...,...,...,...,...,...,...,...
59297,SPAY-0000059298,TX-S-0000059298,POL000024846,NaT,CL00002672,"-9.758,08",Commission Reversal,USD,EFT,Successful,LegacyLedger,False
59401,SPAY-0000059402,TX-S-0000059402,POL000017589,NaT,CL00001185,"7,939.18",Premium,ZAR,EFT,Pending,BankFeed,False
59520,SPAY-0000059521,TX-S-0000059521,POL000025018,NaT,CL00003269,"11,979.93",Commission,ZAR,EFT,Reversed,BankFeed,False
59687,SPAY-0000059688,TX-S-0000059688,POL000017576,NaT,CL00008224,"10,545.55",Commission,ZAR,Debit Order,Reversed,PaymentsHub,False


In [323]:
payments_cleaned['amount'].unique()

<ArrowStringArray>
[    '1,826.09',    '-7,370.85',     '4,318.19',     '4,114.14',
    '21,151.57',            nan, '(-24,653.54)',      '5285.68',
    '20,646.63',   '-21,763.22',
 ...
     '2,172.71',    '22,939.73',      '9983.90',    '19.757,32',
     '19046.91',    '13.273,09',     '9,196.56',     '6.299,68',
     '4,285.22',     '23240.21']
Length: 58441, dtype: str

In [324]:
payments_cleaned['amount'].isnull().sum()

np.int64(359)

In [325]:
payments_cleaned.head(10)

,source_row_id,transaction_ref,policy_id,transaction_date_new,client_id,amount,transaction_type,currency,payment_method,payment_status,source_system,A04_flag
0,SPAY-0000000001,TX-S-0000000001,POL000003392,2022-11-15,CL00007794,"1,826.09",Commission,ZAR,Unknown,Reversed,LegacyLedger,False
1,SPAY-0000000002,TX-S-0000000002,POL000000796,2022-10-23,CL00008081,"-7,370.85",Refund,ZAR,EFT,Successful,PaymentsHub,False
2,SPAY-0000000003,TX-S-0000000003,POL000003511,2025-03-10,CL00007728,"4,318.19",EFT,ZAR,Card,Successful,LegacyLedger,False
3,SPAY-0000000004,TX-S-0000000004,POL000012172,2025-03-21,CL00001221,"4,114.14",Debit Order,ZAR,EFT,Successful,PaymentsHub,False
4,SPAY-0000000005,TX-S-0000000005,POL000014430,2025-04-23,CL00008773,"21,151.57",EFT,ZAR,Debit Order,Reversed,BankFeed,False
5,SPAY-0000000006,TX-S-0000000006,POL000008647,2025-03-09,CL00000156,NaN,Premium,USD,Unknown,Failed,BankFeed,False
6,SPAY-0000000007,TX-S-0000000007,POL000000436,2024-06-23,CL00003805,"(-24,653.54)",Reversal,ZAR,Cash,Successful,PaymentsHub,False
7,SPAY-0000000008,TX-S-0000000008,POL000024244,2022-05-31,CL00000252,5285.68,Premium,ZAR,Cash,Successful,PaymentsHub,False
8,SPAY-0000000009,TX-S-0000000009,POL000012945,2021-03-31,CL00007328,"20,646.63",Premium,ZAR,Debit Order,Reversed,LegacyLedger,False
9,SPAY-0000000010,TX-S-0000000010,POL000010985,2025-10-10,CL00004931,"-21,763.22",Commission Reversal,ZAR,Card,Successful,BankFeed,False


In [326]:
#Create a flag where A16_R1 is displayed. 
payments_cleaned['A16_R1'] = payments_cleaned['amount'].isna()

In [327]:
# Create quarantine dataset
quarantine_A16_R1 = payments_cleaned[payments_cleaned['A16_R1']].copy()

# Keep only valid records
payments_cleaned = payments_cleaned[~payments_cleaned['A16_R1']].copy()

In [328]:
# Checking if the transaction date  is correctly quarantined 
print("Original records:", len(payments_cleaned))
print("Quarantined records:", len(quarantine_A16_R1))
print('Total Records:', len(quarantine_A16_R1)+ len(payments_cleaned))

Original records: 59025
Quarantined records: 359
Total Records: 59384


In [329]:
#Creating a new table for the quarantined A16_R1 values 
quarantine_A16_R1.to_csv("../quarantine/quarantine_A16_R1.csv", index=False)

In [332]:
#Create an A16_R3 flag where amount show "not recorded"
payments_cleaned['A16_R3'] = (
    payments_cleaned['amount']
    .str.strip()
    .str.lower()
    .eq('not recorded')
)

In [334]:
# Create quarantine dataset
quarantine_A16_R3 = payments_cleaned[payments_cleaned['A16_R3']].copy()

# Keep only valid records
payments_cleaned = payments_cleaned[~payments_cleaned['A16_R3']].copy()

In [335]:
# Checking if the transaction date  is correctly quarantined 
print("Original records:", len(payments_cleaned))
print("Quarantined records:", len(quarantine_A16_R3))
print('Total Records:', len(quarantine_A16_R3)+ len(payments_cleaned))

Original records: 58847
Quarantined records: 178
Total Records: 59025


In [336]:
#Creating a new table for the quarantined A16_R3 values 
quarantine_A16_R3.to_csv("../quarantine/quarantine_A16_R3.csv", index=False)